# Machine Learning Zoomcamp 2026 - Homework 2

**Topic:** Linear Regression, Feature Imputation, Regularization (Ridge), and Cross-Validation Stability.

Reference: [Homework 2 Description](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/02-regression/homework.md)

## 1. Imports and Data Preparation

In [1]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df_raw = pd.read_csv(DATA_URL)

features = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year",
    "fuel_efficiency_mpg",
]
df = df_raw[features].copy()
df.head()


,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


## Exploratory Data Analysis (EDA)

Look at the `fuel_efficiency_mpg` distribution.

In [2]:
df['fuel_efficiency_mpg'].describe()


count    10000.000000
mean        29.997700
std          2.930245
min         19.800000
25%         28.000000
50%         30.000000
75%         31.900000
max         41.200000
Name: fuel_efficiency_mpg, dtype: float64

## Q1. Missing Values

There's one column with missing values. What is it?

In [3]:
missing = df.isnull().sum()
print(missing)
print(f"\nColumn with missing values: {missing[missing > 0].index[0]}")


engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

Column with missing values: horsepower


## Q2. Median Horsepower

What's the median (50% percentile) for variable `'horsepower'`?

In [4]:
median_hp = df['horsepower'].median()
print(f"Median of horsepower: {median_hp}")


Median of horsepower: 254.0


## Helper Functions for Linear Regression & RMSE

Implementing Normal Equation with optional Ridge ($L_2$) regularization.

In [5]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X_full = np.column_stack([ones, X])
    XTX = X_full.T.dot(X_full)
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X_full.T).dot(y)
    return w[0], w[1:]

def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X_full = np.column_stack([ones, X])
    XTX = X_full.T.dot(X_full)
    reg = r * np.eye(XTX.shape[0])
    XTX_reg = XTX + reg
    XTX_inv = np.linalg.inv(XTX_reg)
    w = XTX_inv.dot(X_full.T).dot(y)
    return w[0], w[1:]

def rmse(y_true, y_pred):
    error = y_pred - y_true
    mse = (error ** 2).mean()
    return float(np.sqrt(mse))


## Prepare and Split Dataset (Seed 42)

Splitting data into 60% Train, 20% Validation, and 20% Test.

In [6]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train : n_train + n_val]].copy()
df_test = df.iloc[idx[n_train + n_val :]].copy()

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']


## Q3. Missing Value Imputation: 0 vs Mean

Compare validation RMSE when filling missing values with 0 versus the mean (calculated from train set only).

In [7]:
# Option A: Fill with 0
X_train_0 = df_train.fillna(0).values
w0_0, w_0 = train_linear_regression(X_train_0, y_train)
X_val_0 = df_val.fillna(0).values
y_pred_0 = w0_0 + X_val_0.dot(w_0)
rmse_0 = round(rmse(y_val, y_pred_0), 3)

# Option B: Fill with mean of training set
mean_hp = df_train['horsepower'].mean()
X_train_mean = df_train.fillna(mean_hp).values
w0_mean, w_mean = train_linear_regression(X_train_mean, y_train)
X_val_mean = df_val.fillna(mean_hp).values
y_pred_mean = w0_mean + X_val_mean.dot(w_mean)
rmse_mean = round(rmse(y_val, y_pred_mean), 3)

print(f"RMSE with 0:    {rmse_0}")
print(f"RMSE with mean: {rmse_mean}")
print("Better option: ", "With mean" if rmse_mean < rmse_0 else ("With 0" if rmse_0 < rmse_mean else "Both are equally good"))


RMSE with 0:    2.205
RMSE with mean: 2.202
Better option:  With mean


## Q4. Regularized Linear Regression (Tuning $r$)

Filling missing values with 0 and evaluating $r \in [0, 0.01, 0.1, 1, 5, 10, 100]$.

In [8]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]
for r in r_values:
    w0_r, w_r = train_linear_regression_reg(X_train_0, y_train, r=r)
    y_pred_r = w0_r + X_val_0.dot(w_r)
    score = round(rmse(y_val, y_pred_r), 4)
    print(f"r={r:>5} -> RMSE: {score:.4f}")


r=    0 -> RMSE: 2.2053
r= 0.01 -> RMSE: 2.2058
r=  0.1 -> RMSE: 2.2241
r=    1 -> RMSE: 2.3492
r=    5 -> RMSE: 2.4094
r=   10 -> RMSE: 2.4195
r=  100 -> RMSE: 2.4292


**Answer: $r = 0$ (no regularization).**

The lowest validation RMSE is **2.2053**, achieved with $r = 0$. The next closest result is **2.2058** at $r = 0.01$.

Lower RMSE means better predictions. Increasing $r$ increases the validation RMSE in this experiment, so regularization does not improve performance for this dataset and split.

## Q5. Seed Variation and Score Stability

Evaluate model performance across seeds `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]` and compute the standard deviation of RMSE scores.

In [9]:
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
scores = []

for seed in seeds:
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    d_tr = df.iloc[idx[:n_train]].copy()
    d_v = df.iloc[idx[n_train : n_train + n_val]].copy()

    y_tr = d_tr['fuel_efficiency_mpg'].values
    y_v = d_v['fuel_efficiency_mpg'].values

    del d_tr['fuel_efficiency_mpg']
    del d_v['fuel_efficiency_mpg']

    X_tr = d_tr.fillna(0).values
    w0_s, w_s = train_linear_regression(X_tr, y_tr)

    X_v = d_v.fillna(0).values
    y_pr = w0_s + X_v.dot(w_s)
    scores.append(rmse(y_v, y_pr))

std_scores = round(float(np.std(scores)), 3)
print("RMSE scores per seed:", [round(s, 4) for s in scores])
print(f"Standard deviation: {std_scores}")


RMSE scores per seed: [2.2392, 2.2021, 2.1634, 2.2044, 2.2019, 2.2458, 2.2697, 2.1908, 2.2166, 2.207]
Standard deviation: 0.029


## Q6. Training on Full Train (Train + Val) with Seed 9

Combine train and validation datasets with seed 9 and evaluate on test set with $r=0.001$.

In [10]:
np.random.seed(9)
idx_q6 = np.arange(n)
np.random.shuffle(idx_q6)

df_train_q6 = df.iloc[idx_q6[:n_train]].copy()
df_val_q6 = df.iloc[idx_q6[n_train : n_train + n_val]].copy()
df_test_q6 = df.iloc[idx_q6[n_train + n_val :]].copy()

df_full_train = pd.concat([df_train_q6, df_val_q6], axis=0)
y_full_train = df_full_train['fuel_efficiency_mpg'].values
y_test_q6 = df_test_q6['fuel_efficiency_mpg'].values

del df_full_train['fuel_efficiency_mpg']
del df_test_q6['fuel_efficiency_mpg']

X_full_train = df_full_train.fillna(0).values
w0_final, w_final = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

X_test_q6 = df_test_q6.fillna(0).values
y_test_pred = w0_final + X_test_q6.dot(w_final)
test_rmse = round(rmse(y_test_q6, y_test_pred), 3)
print(f"Test RMSE: {test_rmse}")


Test RMSE: 2.236
